# Text Data

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import fitz
def doc_chunker(pdf_path):
    doc = fitz.open(pdf_path)
    pdf_text = ""
    for page in doc:
        pdf_text += page.get_text()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_text(pdf_text)
    keywords = keywords = [
        "soil", "sol",
        "pH",
        "moisture", "humidité",
        "organic matter", "matière organique", "MO",
        "texture", "texture du sol",
        "salinity", "salinité",

        "X", "Y",
        "Longitude", "Longitude",
        "Latitude", "Latitude",
        "Altitude", "Altitude",

        "UD", "Upper Depth", "Profondeur supérieure", "profondeur supérieure",
        "LD", "Lower Depth", "Profondeur inférieure", "profondeur inférieure",
        "Average profondeur", "Profondeur moyenne", "profondeur moyenne",
        "profondeur", "depth",

        "USDA Soil Taxonomy", "Taxonomie des sols USDA",

        "CE", "Conductivité électrique",
        "CEC", "Capacité d’échange cationique",

        "A%", "Argile", "Clay", "taux d’argile",
        "L%", "Limon", "Silt", "taux de limon",
        "S%", "Sable", "Sand", "taux de sable",

        "ALR(L)", "ALR(S)",
        "OM", "matière organique",
    ]
    
    relevant_chunks = [chunk for chunk in chunks if any(k in chunk.lower() for k in keywords)]
    len(relevant_chunks)
    return relevant_chunks

In [3]:
import os
def find_all_pdf_files(root_folder):
    pdf_files = []
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(('.pdf')):
                pdf_files.append(os.path.join(root, file))
    return pdf_files

In [14]:
all_files = find_all_pdf_files("../theses/")
for index , i in enumerate(all_files[200:]):
  print(f"{200+index}/{len(all_files)}")
  text = " ".join(doc_chunker(i))
  Prompt = f"""
    You are an expert in soil data extraction. Given a raw text report, your task is to extract structured data about soil characteristics along with their geographic location.

    ### Text:
    {text}

    ### Instructions:
    - Parse the text and extract the following fields for **each unique location** (based on Latitude and Longitude) and **each reported depth**:
      - filename
      - X
      - Y
      - Longitude
      - Latitude
      - Altitude
      - UD (Upper Depth)
      - LD (Lower Depth)
      - Average profondeur (average of UD and LD)
      - USDA Soil Taxonomy
      - pH
      - CE
      - CEC
      - A% (Argile / Clay %)
      - L% (Limon / Silt %)
      - S% (Sable / Sand %)
      - ALR(L)
      - ALR(S)
      - OM (Organic Matter)

    - If a field is missing in the text, set its value to `null`.
    - Your output must be in **CSV format** and contain only the table — no explanations, no markdown formatting.

    ### Output format:
    Each row should represent a single depth profile. The first line should be the header:

    filename,X,Y,Longitude,Latitude,Altitude,UD,LD,Average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM

    Followed by one row per depth/location.

    Note that the file name is: {i}
    """

  from together import Together

  client = Together(api_key="875fef17f5f34497577741f100d62abc27b7a727ece6cb92a3f084d2f05dfb40")

  response = client.chat.completions.create(
      model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
      messages=[{"role": "user", "content": Prompt}]
  )
  print(response.choices[0].message.content)
  with open("res3.txt" , "a" , encoding="utf8") as src:
      src.write(response.choices[0].message.content)


200/218


KeyboardInterrupt: 

In [1]:
import pandas as pd 
df = pd.read_csv("res.csv")
df

,filename,X,Y,Longitude,Latitude,Altitude,UD,LD,Average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,../theses/Nord\memoires traited\Bio écologie d...,NaN,NaN,NaN,NaN,NaN,0,10,NaN,NaN,8.33,NaN,NaN,17.6,20.3,62.1,NaN,NaN,2.12
1,../theses/Nord\memoires traited\Bio écologie d...,NaN,NaN,NaN,NaN,NaN,0,30,NaN,NaN,8.29,NaN,NaN,42.7,31.8,25.5,NaN,NaN,1.59
2,../theses/Nord\memoires traited\Bio écologie d...,NaN,NaN,NaN,NaN,NaN,0,30,NaN,NaN,8.3,NaN,NaN,31.6,23,45.2,NaN,NaN,1.05
3,../theses/Nord\memoires traited\Caractérisatio...,NaN,NaN,3.0583,36.7232,1200,0,10,5,NaN,7.2,NaN,15,20,40,40,NaN,NaN,NaN
4,../theses/Nord\memoires traited\Caractérisatio...,NaN,NaN,3.0583,36.7232,1200,10,20,15,NaN,7.5,NaN,12,25,35,40,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,0,8,4,NaN,7.39,2.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2352,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,8,34,21,NaN,7.65,4.87,NaN,NaN,NaN,NaN,NaN,NaN,0.53
2353,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,34,43,38.5,NaN,7.99,5.48,NaN,NaN,NaN,NaN,NaN,NaN,1.06
2354,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,43,60,51.5,NaN,8.04,4.07,NaN,NaN,NaN,NaN,NaN,NaN,0


In [2]:
df_cleaned = df.dropna(subset=["X" , "Y" , "Longitude" , "Latitude"], how='all')
df_cleaned

,filename,X,Y,Longitude,Latitude,Altitude,UD,LD,Average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
3,../theses/Nord\memoires traited\Caractérisatio...,NaN,NaN,3.0583,36.7232,1200,0,10,5,NaN,7.2,NaN,15,20,40,40,NaN,NaN,NaN
4,../theses/Nord\memoires traited\Caractérisatio...,NaN,NaN,3.0583,36.7232,1200,10,20,15,NaN,7.5,NaN,12,25,35,40,NaN,NaN,NaN
57,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,3°0'57.627E,36°48'53.426''N,NaN,NaN,NaN,NaN,NaN,NaN,8.04,0.575,NaN,NaN,NaN,NaN,NaN,NaN
58,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,3°56'12.638''E,36°52'42.096''N,NaN,NaN,NaN,NaN,NaN,NaN,7.65,0.582,NaN,NaN,NaN,NaN,NaN,NaN
59,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,6°10'26.081''E,35°33'39.687''N,NaN,NaN,NaN,NaN,NaN,NaN,7.74,0.210,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,0,8,4,NaN,7.39,2.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2352,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,8,34,21,NaN,7.65,4.87,NaN,NaN,NaN,NaN,NaN,NaN,0.53
2353,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,34,43,38.5,NaN,7.99,5.48,NaN,NaN,NaN,NaN,NaN,NaN,1.06
2354,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.43190,27.19314,NaN,43,60,51.5,NaN,8.04,4.07,NaN,NaN,NaN,NaN,NaN,NaN,0


In [3]:
for i in range(len(df_cleaned)):
    print(f" X : {df_cleaned.iloc[i]['X']} , Y : {df_cleaned.iloc[i]['Y']}")

 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : 2.14 , Y : 30.43
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : 3.15987 , Y : 36.71253
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y : nan
 X : nan , Y 

In [4]:
import re

def converter(value , flag=False):
    if flag : 
        value = value.replace(",",".").replace("′","'").replace("″",'"').replace("’’",'"').replace("’","'").replace("N","").replace("/","").replace("\\","")
    value = value.replace(",",".")
    print(value)
    regex_dms = re.compile(r'^(\d{1,3})[º°\s]+(\d{1,2})[\'\s]+([\d.,]+)[\'\s]*(E|W|N|S)?$')
    match = regex_dms.match(value.strip())
    if match:
        new_value = dms_str_to_decimal(value, match.groups())
        print(new_value)
        return new_value
    regex_dms_no_spaces = re.compile(r'^(\d{1,3})º(\d{1,2})\'([\d.,]+)\'\'?(E|W|N|S)?$')
    match_no_spaces = regex_dms_no_spaces.match(value.strip())
    if match_no_spaces:
        new_value = dms_str_to_decimal(value, match_no_spaces.groups())
        print(new_value)
        return new_value

    return None

def dms_str_to_decimal(dms_str, match_groups):
    degrees, minutes, seconds = map(float, match_groups[:3])
    decimal_value = degrees + (minutes / 60) + (seconds / 3600)
    if match_groups[3]:
        direction = match_groups[3].upper()
        if direction in ['S', 'W']:
            decimal_value = -decimal_value
    return decimal_value


In [5]:
df_cleaned['X'] = df_cleaned['X'].astype(str)
df_cleaned["X"] = df_cleaned["X"].apply(converter)
df_cleaned["X"].value_counts().sum()

nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
2.14
3.15987
3.15987
3.15987
3.15987
3.15987
3.15987
3.15987
3.15987
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
06 06 22
6.106111111111111
06 06 15
6.104166666666666
06 06 35
6.109722222222222
06 06 55
6.115277777777777
06 06 25
6.106944444444444
06 06 35
6.109722222222222
06 06 05
6.101388888888889
06 06 25
6.106944444444444
05 57 14.21
5.953947222222222
05 57 19.15
5.955319444444445
05 56 44.55
5.945708333333333
05 57 29.05
5.958069444444445
05 56 59.38
5.949827777777778
05 57 14.21
5.953947222222222
05 56 34.65
5.942958333333333
05 57 43.88
5.962188888888889
05 58 28.37
5.974547222222222
06 06 22
6.106111111111111
06 06 15
6.104166666666666
06 06 35
6.109722222222222
06 06 55
6.115277777777777
06 06 25
6.106944444444444
06 06 35
6.109722222222222
06 06 05
6.101388888888889
06 06 25
6.106944444444444
05 57 14.21
5.953947222222222
05 57 19.15

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2396268829.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['X'] = df_cleaned['X'].astype(str)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2396268829.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["X"] = df_cleaned["X"].apply(converter)


129

In [6]:
df_cleaned['Y'] = df_cleaned['Y'].astype(str)
df_cleaned["X"] = df_cleaned["Y"].apply(converter)
df_cleaned["X"].value_counts().sum()

nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
30.43
36.71253
36.71253
36.71253
36.71253
36.71253
36.71253
36.71253
36.71253
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
35 34 58
35.58277777777778
35 34 58
35.58277777777778
35 34 48
35.580000000000005
35 34 38
35.577222222222225
35 34 28
35.574444444444445
35 34 41
35.57805555555556
35 34 46
35.57944444444445
35 34 36
35.57666666666667
35 33 42.15
35.56170833333333
35 34 10.03
35.569452777777784
35 34 14.34
35.57065
35 34 14.34
35.57065
35 34 26.39
35.573997222222225
35 34 46.48
35.57957777777778
35 35 02.57
35.584047222222225
35 35 34.72
35.59297777777778
35 35 14.64
35.5874
35 34 58
35.58277777777778
35 34 58
35.58277777777778
35 34 48
35.580000000000005
35 34 38
35.577222222222225
35 34 28
35.574444444444445
35 34 41
35.57805555555556
35 34 46
35.57944444444445
35 34 36
35.57666666666667
35 33 42.15
35.56170833333333
35 34 10.03
35.569452

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\1956847533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Y'] = df_cleaned['Y'].astype(str)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\1956847533.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["X"] = df_cleaned["Y"].apply(converter)


129

In [7]:
def dms_to_dd(degrees, minutes=0, seconds=0):
    return float(degrees) + float(minutes)/60 + float(seconds)/3600

def convert_line_to_decimal(line):
    line = line.strip().replace('˚', '°').replace("''", '"').replace("’", "'").replace("“", '"').replace("”", '"')
    dms_pattern = re.compile(r"(\d+)[°:\s]+(\d+)?[':\s]*(\d+(?:\.\d+)?)?[\"\s]*")
    
    matches = dms_pattern.findall(line)
    if matches:
        degrees, minutes, seconds = 0, 0, 0
        for match in matches:
            if match[0]: degrees = float(match[0])
            if match[1]: minutes = float(match[1])
            if match[2]: seconds = float(match[2])
        return round(dms_to_dd(degrees, minutes, seconds), 8)

    try:
        return round(float(line), 8)
    except:
        return line

In [8]:
df_cleaned['Latitude'] = df_cleaned['Latitude'].astype(str)
df_cleaned['Latitude'] = df_cleaned['Latitude'].apply(convert_line_to_decimal)
for i in df_cleaned["Latitude"]:
    print(i)

36.7232
36.7232
36.81484056
36.87836
35.56102417
32.49022472
32.49022472
36.74718694
36.70705806
52.39416667
36.15978667
30.43N
36.7
36.7
36.7
36.7
36.7
36.7
36.7
36.7
34.716667
34.716667
34.716667
34.716667
34.716667
34.716667
34.633333
34.633333
34.633333
34.633333
34.633333
34.633333
34.716667
34.716667
34.716667
34.716667
34.716667
34.716667
34.8
34.8
34.8
34.8
34.8
34.8
34.7
34.7
34.7
34.683333
34.683333
34.683333
34.9
34.9
34.9
34.916667
34.916667
34.916667
35.582778
35.582778
35.58
35.577222
35.574444
35.578056
35.579444
35.576667
35.561708
35.569453
35.57065
35.57065
35.573997
35.579578
35.584047
35.593
35.5874
35.582778
35.582778
35.58
35.577222
35.574444
35.578056
35.579444
35.576667
35.561708
35.569453
35.57065
35.57065
35.573997
35.579578
35.584047
35.593
35.5874
35.616667
35.6261111
35.6
35.6
35.59
35.59
35.601
35.601
35.603
35.603
35.606
35.606
35.608
35.608
35.61
35.61
35.613
35.613
35.61
35.61
35.576613
35.576613
35.59
35.59
35.59
35.63
35.63
35.63
35.63
35.63
35.6308
3

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2585741323.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Latitude'] = df_cleaned['Latitude'].astype(str)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2585741323.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Latitude'] = df_cleaned['Latitude'].apply(convert_line_to_decimal)


In [9]:
df_cleaned['Longitude'] = df_cleaned['Longitude'].astype(str)
df_cleaned['Longitude'] = df_cleaned['Longitude'].apply(convert_line_to_decimal)
for i in df_cleaned["Longitude"]:
    print(i)

3.0583
3.0583
3.0160075
3.93684389
6.17391139
3.67384111
3.67384111
3.19043194
3.18620333
2.92758333
5.41751889
2.14E
3.15
3.15
3.15
3.15
3.15
3.15
3.15
3.15
6.283333
6.283333
6.283333
6.3
6.3
6.3
5.133333
5.133333
5.133333
5.133333
5.133333
5.133333
5.366667
5.366667
5.366667
5.366667
5.366667
5.366667
5.683333
5.683333
5.683333
5.716667
5.716667
5.716667
6.433333
6.433333
6.433333
6.45
6.45
6.45
5.633333
5.633333
5.633333
5.633333
5.633333
5.633333
6.106111
6.104167
6.109722
6.115278
6.106944
6.109722
6.101389
6.106944
5.953947
5.955319
5.945708
5.958069
5.949828
5.953947
5.942958
5.962189
5.974547
6.106111
6.104167
6.109722
6.115278
6.106944
6.109722
6.101389
6.106944
5.953947
5.955319
5.945708
5.958069
5.949828
5.953947
5.942958
5.962189
5.974547
6.233333
6.2388889
6.25
6.25
6.212
6.212
6.223
6.223
6.229
6.229
6.235
6.235
6.24
6.24
6.247
6.247
6.27
6.27
6.267
6.267
6.188585
6.188585
6.11
6.11
6.11
6.04
6.04
6.04
6.04
6.04
6.1885
6.1885
6.1885
6.1885
6.1722
6.1722
6.1722
5.883
5.883

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2716947769.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Longitude'] = df_cleaned['Longitude'].astype(str)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\2716947769.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Longitude'] = df_cleaned['Longitude'].apply(convert_line_to_decimal)


In [10]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 3 to 2355
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   filename            1636 non-null   object 
 1   X                   129 non-null    float64
 2   Y                   1636 non-null   object 
 3   Longitude           1381 non-null   object 
 4   Latitude            1399 non-null   object 
 5   Altitude            989 non-null    object 
 6   UD                  1519 non-null   object 
 7   LD                  1437 non-null   object 
 8   Average profondeur  1309 non-null   object 
 9   USDA Soil Taxonomy  583 non-null    object 
 10  pH                  1324 non-null   object 
 11  CE                  1221 non-null   object 
 12  CEC                 489 non-null    object 
 13  A%                  1139 non-null   object 
 14  L%                  1130 non-null   object 
 15  S%                  1153 non-null   object 
 16  ALR(L)     

In [22]:
def type_converter(val):
    try : 
        return round(float(val), 2)
    except Exception:
        return None 

In [12]:
df_cleaned["Y"] = df_cleaned["Y"].apply(type_converter)
df_cleaned["Latitude"] = df_cleaned["Latitude"].apply(type_converter)
df_cleaned["Longitude"] = df_cleaned["Longitude"].apply(type_converter)

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\695684658.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["Y"] = df_cleaned["Y"].apply(type_converter)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\695684658.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["Latitude"] = df_cleaned["Latitude"].apply(type_converter)
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\695684658.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usi

In [13]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 3 to 2355
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   filename            1636 non-null   object 
 1   X                   129 non-null    float64
 2   Y                   321 non-null    float64
 3   Longitude           1376 non-null   float64
 4   Latitude            1370 non-null   float64
 5   Altitude            989 non-null    object 
 6   UD                  1519 non-null   object 
 7   LD                  1437 non-null   object 
 8   Average profondeur  1309 non-null   object 
 9   USDA Soil Taxonomy  583 non-null    object 
 10  pH                  1324 non-null   object 
 11  CE                  1221 non-null   object 
 12  CEC                 489 non-null    object 
 13  A%                  1139 non-null   object 
 14  L%                  1130 non-null   object 
 15  S%                  1153 non-null   object 
 16  ALR(L)     

In [14]:
df_cleaned['Latitude'] = df_cleaned['Latitude'].fillna(df['X'])
df_cleaned['Longitude'] = df_cleaned['Longitude'].fillna(df['Y'])

C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\1819996865.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Latitude'] = df_cleaned['Latitude'].fillna(df['X'])
C:\Users\abdo7\AppData\Local\Temp\ipykernel_2776\1819996865.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['Longitude'] = df_cleaned['Longitude'].fillna(df['Y'])


In [15]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 3 to 2355
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   filename            1636 non-null   object 
 1   X                   129 non-null    float64
 2   Y                   321 non-null    float64
 3   Longitude           1595 non-null   object 
 4   Latitude            1609 non-null   object 
 5   Altitude            989 non-null    object 
 6   UD                  1519 non-null   object 
 7   LD                  1437 non-null   object 
 8   Average profondeur  1309 non-null   object 
 9   USDA Soil Taxonomy  583 non-null    object 
 10  pH                  1324 non-null   object 
 11  CE                  1221 non-null   object 
 12  CEC                 489 non-null    object 
 13  A%                  1139 non-null   object 
 14  L%                  1130 non-null   object 
 15  S%                  1153 non-null   object 
 16  ALR(L)     

In [16]:
df_unique = df_cleaned.drop_duplicates(subset=["Latitude" , "Longitude"])
df_unique

,filename,X,Y,Longitude,Latitude,Altitude,UD,LD,Average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
3,../theses/Nord\memoires traited\Caractérisatio...,NaN,NaN,3.0583,36.7232,1200,0,10,5,NaN,7.2,NaN,15,20,40,40,NaN,NaN,NaN
57,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,3.016008,36.814841,NaN,NaN,NaN,NaN,NaN,NaN,8.04,0.575,NaN,NaN,NaN,NaN,NaN,NaN
58,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,3.936844,36.87836,NaN,NaN,NaN,NaN,NaN,NaN,7.65,0.582,NaN,NaN,NaN,NaN,NaN,NaN
59,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,6.173911,35.561024,NaN,NaN,NaN,NaN,NaN,NaN,7.74,0.210,NaN,NaN,NaN,NaN,NaN,NaN
60,../theses/Nord\memoires traited\TH9217.pdf,NaN,NaN,3.673841,32.490225,NaN,NaN,NaN,NaN,NaN,NaN,7.17,2.93,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2337,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.246503,27.352776,NaN,0,20,10,NaN,7.63,1.15,NaN,NaN,NaN,NaN,NaN,NaN,2.38
2341,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,44.7217,21.1321,NaN,0,75,37.5,NaN,7.65,8.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2343,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.4069,27.95762,NaN,0,10,5,NaN,6.72,3.71,NaN,NaN,NaN,NaN,NaN,NaN,1.06
2348,../theses/Sud\new memoires\In salah\TIATAOUI_T...,NaN,NaN,2.4464,27.1988,NaN,0,3,1.5,NaN,6.8,3.75,NaN,NaN,NaN,NaN,NaN,NaN,1.32


In [17]:
df_cleaned.to_csv("mydata.csv")

In [18]:
df1 = pd.read_excel("../results/soil_dataset.xlsx",sheet_name="soil_data")
df1 = df1.drop(columns=["id" ,"id_memoire","id_ech","Wilaya","Region"])
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1256 entries, 0 to 1255
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   X                   1256 non-null   float64
 1   Y                   1256 non-null   float64
 2   altitude            759 non-null    object 
 3   UD                  1256 non-null   int64  
 4   LD                  1256 non-null   int64  
 5   average profondeur  1256 non-null   float64
 6   USDA Soil Taxonomy  241 non-null    object 
 7   pH                  1256 non-null   float64
 8   CE                  1256 non-null   float64
 9   CEC                 1256 non-null   float64
 10  A%                  1256 non-null   float64
 11  L%                  1256 non-null   float64
 12  S%                  1256 non-null   float64
 13  ALR(L)              1088 non-null   float64
 14  ALR(S)              1088 non-null   float64
 15  OM                  1256 non-null   float64
dtypes: flo

In [29]:
df2 = pd.read_csv("mydata.csv")
df2 = df2.drop(columns=["filename" , "Unnamed: 0" , "X" , "Y"])
df2["Latitude"] = df2["Latitude"].apply(type_converter)
df2["Longitude"] = df2["Longitude"].apply(type_converter)
df2

,Longitude,Latitude,Altitude,UD,LD,Average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.06,36.72,1200,0,10,5,NaN,7.2,NaN,15.000,20,40,40,NaN,NaN,NaN
1,3.06,36.72,1200,10,20,15,NaN,7.5,NaN,12.000,25,35,40,NaN,NaN,NaN
2,3.02,36.81,NaN,NaN,NaN,NaN,NaN,NaN,8.04,0.575,NaN,NaN,NaN,NaN,NaN,NaN
3,3.94,36.88,NaN,NaN,NaN,NaN,NaN,NaN,7.65,0.582,NaN,NaN,NaN,NaN,NaN,NaN
4,6.17,35.56,NaN,NaN,NaN,NaN,NaN,NaN,7.74,0.210,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1631,2.43,27.19,NaN,0,8,4,NaN,7.39,2.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1632,2.43,27.19,NaN,8,34,21,NaN,7.65,4.87,NaN,NaN,NaN,NaN,NaN,NaN,0.53
1633,2.43,27.19,NaN,34,43,38.5,NaN,7.99,5.48,NaN,NaN,NaN,NaN,NaN,NaN,1.06
1634,2.43,27.19,NaN,43,60,51.5,NaN,8.04,4.07,NaN,NaN,NaN,NaN,NaN,NaN,0


In [30]:
for i in df2.columns:
    if i not in ["USDA Soil Taxonomy"]:
        df2[i] = df2[i].apply(type_converter)
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Longitude           1555 non-null   float64
 1   Latitude            1563 non-null   float64
 2   Altitude            951 non-null    float64
 3   UD                  1488 non-null   float64
 4   LD                  1433 non-null   float64
 5   Average profondeur  1302 non-null   float64
 6   USDA Soil Taxonomy  583 non-null    object 
 7   pH                  1277 non-null   float64
 8   CE                  1184 non-null   float64
 9   CEC                 489 non-null    float64
 10  A%                  1101 non-null   float64
 11  L%                  1097 non-null   float64
 12  S%                  1114 non-null   float64
 13  ALR(L)              151 non-null    float64
 14  ALR(S)              95 non-null     float64
 15  OM                  852 non-null    float64
dtypes: flo

In [31]:
df2.columns = df1.columns 
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1636 entries, 0 to 1635
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   X                   1555 non-null   float64
 1   Y                   1563 non-null   float64
 2   altitude            951 non-null    float64
 3   UD                  1488 non-null   float64
 4   LD                  1433 non-null   float64
 5   average profondeur  1302 non-null   float64
 6   USDA Soil Taxonomy  583 non-null    object 
 7   pH                  1277 non-null   float64
 8   CE                  1184 non-null   float64
 9   CEC                 489 non-null    float64
 10  A%                  1101 non-null   float64
 11  L%                  1097 non-null   float64
 12  S%                  1114 non-null   float64
 13  ALR(L)              151 non-null    float64
 14  ALR(S)              95 non-null     float64
 15  OM                  852 non-null    float64
dtypes: flo

In [32]:
df2

,X,Y,altitude,UD,LD,average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.06,36.72,1200.0,0.0,10.0,5.0,NaN,7.20,NaN,15.00,20.0,40.0,40.0,NaN,NaN,NaN
1,3.06,36.72,1200.0,10.0,20.0,15.0,NaN,7.50,NaN,12.00,25.0,35.0,40.0,NaN,NaN,NaN
2,3.02,36.81,NaN,NaN,NaN,NaN,NaN,NaN,8.04,0.57,NaN,NaN,NaN,NaN,NaN,NaN
3,3.94,36.88,NaN,NaN,NaN,NaN,NaN,NaN,7.65,0.58,NaN,NaN,NaN,NaN,NaN,NaN
4,6.17,35.56,NaN,NaN,NaN,NaN,NaN,NaN,7.74,0.21,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1631,2.43,27.19,NaN,0.0,8.0,4.0,NaN,7.39,2.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1632,2.43,27.19,NaN,8.0,34.0,21.0,NaN,7.65,4.87,NaN,NaN,NaN,NaN,NaN,NaN,0.53
1633,2.43,27.19,NaN,34.0,43.0,38.5,NaN,7.99,5.48,NaN,NaN,NaN,NaN,NaN,NaN,1.06
1634,2.43,27.19,NaN,43.0,60.0,51.5,NaN,8.04,4.07,NaN,NaN,NaN,NaN,NaN,NaN,0.00


In [27]:
df_combined = pd.concat([df1, df2], ignore_index=True)
df_combined

,X,Y,altitude,UD,LD,average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.10700,36.684000,22,15.0,30.0,22.5,NaN,7.870,0.170,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019
1,3.10700,36.684000,22,30.0,60.0,45.0,NaN,7.774,0.170,17.207000,40.361496,37.281777,22.356727,-0.034471,-0.256559,1.503
2,3.10700,36.684000,22,60.0,100.0,80.0,NaN,7.799,0.170,15.049000,49.608530,42.317043,8.074427,-0.069041,-0.788445,0.706
3,3.13333,36.071667,24,15.0,30.0,22.5,CL,7.950,0.260,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645
4,3.13333,36.071667,24,30.0,60.0,45.0,CL,8.096,0.279,40.610953,33.898341,39.248386,26.853273,0.063643,-0.101181,1.097
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2887,2.43000,27.190000,NaN,0.0,8.0,4.0,NaN,7.390,2.860,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2888,2.43000,27.190000,NaN,8.0,34.0,21.0,NaN,7.650,4.870,NaN,NaN,NaN,NaN,NaN,NaN,0.530
2889,2.43000,27.190000,NaN,34.0,43.0,38.5,NaN,7.990,5.480,NaN,NaN,NaN,NaN,NaN,NaN,1.060
2890,2.43000,27.190000,NaN,43.0,60.0,51.5,NaN,8.040,4.070,NaN,NaN,NaN,NaN,NaN,NaN,0.000


In [28]:
df_unique = df_combined.drop_duplicates(subset=["X" , "Y"])
df_unique

,X,Y,altitude,UD,LD,average profondeur,USDA Soil Taxonomy,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM
0,3.107000,36.684000,22,15.0,30.0,22.5,NaN,7.87,0.170000,17.929000,46.172091,45.613249,8.214660,-0.005289,-0.749790,2.019
3,3.133330,36.071667,24,15.0,30.0,22.5,CL,7.95,0.260000,40.610953,32.330400,40.460456,27.209144,0.097420,-0.074896,2.645
5,3.150000,36.724167,42,15.0,30.0,22.5,NaN,8.50,2.632808,40.610953,13.000000,32.000000,55.000000,0.391206,0.626419,7.270
7,2.880278,36.757500,44,5.0,15.0,10.0,NaN,8.59,0.370000,40.610953,7.210000,11.570000,81.220000,0.205397,1.051727,4.100
9,3.178361,36.704083,12,15.0,30.0,22.5,NaN,7.83,2.632808,3.750000,27.239966,42.857252,29.902781,NaN,NaN,1.360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2873,2.250000,27.350000,NaN,0.0,20.0,10.0,NaN,7.63,1.150000,NaN,NaN,NaN,NaN,NaN,NaN,2.380
2877,44.720000,21.130000,NaN,0.0,75.0,37.5,NaN,7.65,8.650000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2879,2.410000,27.960000,NaN,0.0,10.0,5.0,NaN,6.72,3.710000,NaN,NaN,NaN,NaN,NaN,NaN,1.060
2884,2.450000,27.200000,NaN,0.0,3.0,1.5,NaN,6.80,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,1.320


In [33]:
df_combined.to_csv("CombinedData.csv")

In [5]:
from pyproj import Transformer
import pandas as pd

In [11]:
df = pd.read_csv("../results/CombinedData.csv")
transformer = Transformer.from_crs("EPSG:30491", "EPSG:4326", always_xy=True)

for i in range(len(df)):
    if df.iloc[i]["X"] > 100 and df.iloc[i]["X"] < 1000 and df.iloc[i]["Y"] > 100 and df.iloc[i]["Y"]<1000:
        x = df.at[i, "X"] * 1000
        y = df.at[i, "Y"] * 1000
        lon, lat = transformer.transform(x, y)
        df.at[i, "X"] = round(lon, 2) 
        df.at[i, "Y"] = round(lat, 2) 
        print(df.at[i, "X"] , df.at[i, "Y"])
    elif df.iloc[i]["X"] > 1000 and df.iloc[i]["Y"] > 1000:
        lon, lat = transformer.transform(x, y)
        df.at[i, "X"] = round(lon, 2) 
        df.at[i, "Y"] = round(lat, 2) 
        print(df.at[i, "X"] , df.at[i, "Y"])

2.03 35.94
0.75 37.88
0.75 37.88
0.67 37.85
0.67 37.85
0.75 37.92
0.74 37.92
0.74 37.92
0.74 37.92
0.63 37.87
0.63 37.87
0.63 37.87
0.74 37.88
0.74 37.88
0.75 37.92
0.75 37.92
0.72 37.89
0.72 37.89
0.75 37.92
0.75 37.92
0.75 37.92
0.79 37.96
0.79 37.96
0.79 37.96
0.74 37.91
0.74 37.91
0.77 37.97
0.77 37.97
0.77 37.97
0.76 37.88
0.76 37.88
0.76 37.88
0.76 37.88
0.76 37.88
0.66 37.96
0.66 37.96
0.69 37.95
0.69 37.95
0.69 37.95
0.69 37.95
0.68 37.92
0.68 37.92
0.68 37.92
0.66 37.93
0.66 37.93
0.66 37.93
0.66 37.93
0.76 37.98
0.76 37.98
0.76 37.98
0.74 37.97
0.74 37.97
0.74 37.97
0.74 37.97
0.61 37.84
0.61 37.84
1.96 35.86
1.96 35.86
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94
2.03 35.94

In [12]:
df.to_csv("CombinedData.csv")